In [0]:
%pip install deltalake

In [0]:
%run ../utils/feat_squad2_99_helpers

In [0]:
def get_squad3_client():
    return get_adls_client().get_file_system_client("squad3")
 
def ler_delta_silver_squad3(nome_tabela: str) -> "pyspark.sql.DataFrame":
    """
    Versao segura para tabelas grandes: converte em lotes pequenos e uniao
    incremental no Spark, em vez de acumular TODOS os arquivos em pandas
    antes de converter (o que estourava memoria do driver em tabelas com
    muitas linhas, como itens_pedido).
    """
    container_client = get_squad3_client()
    path_silver       = f"silver/{nome_tabela}"
 
    log.info(f"Lendo Silver (squad3): {nome_tabela}")
 
    paths = [
        p.name
        for p in container_client.get_paths(path=path_silver, recursive=True)
        if p.name.endswith(".parquet") and "_delta_log" not in p.name
    ]
 
    if not paths:
        raise FileNotFoundError(f"Nenhum arquivo encontrado em {path_silver}")
 
    log.info(f"{len(paths)} arquivo(s) parquet encontrado(s)")
 
    BATCH_SIZE = 20
    df_final = None
    lote_pandas = []
 
    for i, file_path in enumerate(paths, start=1):
        file_client = container_client.get_file_client(file_path)
        data        = file_client.download_file().readall()
        lote_pandas.append(pd.read_parquet(io.BytesIO(data)))
 
        if len(lote_pandas) >= BATCH_SIZE or i == len(paths):
            df_lote_pd    = pd.concat(lote_pandas, ignore_index=True)
            df_lote_spark = spark.createDataFrame(df_lote_pd)
            df_final = df_lote_spark if df_final is None else df_final.union(df_lote_spark)
            log.info(f"Progresso: {i}/{len(paths)} arquivos processados")
            lote_pandas = []
 
    return df_final

In [0]:
container_client = get_squad3_client()
 
todos_arquivos = [
    item.name for item in container_client.get_paths(path="silver", recursive=True)
    if not item.is_directory
]
 
pastas = set()
for caminho in todos_arquivos:
    partes = caminho.split("/")
    if len(partes) >= 2 and partes[0] == "silver":
        pastas.add(partes[1])
 
log.info(f"Tabelas encontradas em squad3/silver/: {sorted(pastas)}")

In [0]:
NOME_TABELA_ITENS = "ecommerce_itens_pedido"  # nome real confirmado via listagem do storage
 
df_itens_raw = ler_delta_silver_squad3(NOME_TABELA_ITENS)
log.info(f"Total bruto lido: {df_itens_raw.count():,}")
df_itens_raw.printSchema()
 
# COMMAND ----------
 
# DBTITLE 1,PASSO 2 - Existe a mesma duplicacao 4x?
from pyspark.sql.functions import col, count as spark_count
 
total_linhas = df_itens_raw.count()
 
# Ajustar a coluna de chave conforme o schema real (id_item ou combinacao id_pedido+produto)
colunas = df_itens_raw.columns
log.info(f"Colunas disponiveis: {colunas}")
 
if "id_item" in colunas:
    chave = "id_item"
elif "id_item_pedido" in colunas:
    chave = "id_item_pedido"
else:
    chave = None
    log.warning("Nenhuma coluna de chave unica obvia encontrada - inspecionar colunas acima manualmente.")
 
if chave:
    total_distintos = df_itens_raw.select(chave).distinct().count()
    duplicados = total_linhas - total_distintos
    log.info(f"Total de linhas       : {total_linhas:,}")
    log.info(f"{chave} distintos      : {total_distintos:,}")
    log.info(f"Linhas duplicadas      : {duplicados:,}  ({round(100*duplicados/total_linhas,2)}%)")

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import row_number
 
df_pedidos_raw = ler_delta_silver_squad3("ecommerce_pedidos")
w = Window.partitionBy("id_pedido").orderBy(col("silver_processed_at").desc())
df_pedidos = df_pedidos_raw.withColumn("_rn", row_number().over(w)).filter(col("_rn") == 1).drop("_rn")
 
ids_pedidos = set(r["id_pedido"] for r in df_pedidos.select("id_pedido").distinct().collect())
ids_itens   = set(r["id_pedido"] for r in df_itens_raw.select("id_pedido").distinct().collect())
 
intersecao = ids_pedidos & ids_itens
cobertura  = round(100 * len(intersecao) / len(ids_pedidos), 2)
 
log.info(f"id_pedido em ecommerce_pedidos (squad3) : {len(ids_pedidos):,}")
log.info(f"id_pedido distintos em itens_pedido     : {len(ids_itens):,}")
log.info(f"Cobertura (pedidos com itens correspondentes) : {cobertura}%")
 
if cobertura > 95:
    log.info("VIAVEL: cobertura alta - podemos usar itens_pedido como fonte de features.")
elif cobertura > 50:
    log.warning("PARCIAL: cobertura media - usar com cautela, pode introduzir muitos nulos.")
else:
    log.warning("BAIXA COBERTURA: pouco aproveitamento - reavaliar uso desta tabela.")